# Writer Palmyra Vision on Amazon Bedrock Mantle

Writer's Palmyra Vision model on the `bedrock-mantle` endpoint. This is a deliberately
instructive case: it is vision-capable but **does not support tool calling**, so it
shows both a specialised strength and how to work around a real limitation.

**Models covered in this notebook**

| Model ID | Notes |
|---|---|
| `writer.palmyra-vision-7b` | Vision-language, 7B. Probe tool support (§7) |

## Which API? Chat Completions.
This family is served by the **OpenAI-compatible Chat Completions API** on the
`bedrock-mantle` endpoint, at the bare `/v1` path. The Responses API returns
**400 "does not support this API"** for these models — we prove that in §2 rather
than asking you to take it on trust.

## Self-contained, but see also
Everything you need is here. For deeper background on shared mechanics:
- **Auth (SigV4 (AWS Signature Version 4) + short-term API keys), the three URL paths,
  model discovery** →
  `../00-foundations/01-endpoints-auth-and-the-three-paths.ipynb`
- **Projects, cost attribution, data retention / ZDR (zero data retention), CloudWatch
  namespace** →
  `../00-foundations/02-governance-projects-and-retention.ipynb`
- **Quotas, retry/backoff, service tiers, TTFT (time-to-first-token) measurement** →
  `../00-foundations/03-scaling-tiers-and-latency.ipynb`

## Prerequisites
```bash
pip install -r ../requirements.txt
```

Needs openai, aws-bedrock-token-generator.

`requirements.txt` pins the exact versions this collection was tested
against. An unpinned install resolves whatever is current, which may be
untested or compromised (OWASP LLM03, Supply Chain).

In [1]:
import json
import sys
import time

sys.path.insert(0, "../_shared")
from bedrock import err, parse_json_lenient, post, safe_print, ttft

REGION = "us-east-1"

VISION = "writer.palmyra-vision-7b"


# Chat-Completions families live at the BARE /v1 path — not /openai/v1
# (that prefix is only for gemma-4, gpt-5.x and grok). See ../00-foundations/01.
PREFIX = "/v1"
BASE_URL = f"https://bedrock-mantle.{REGION}.api.aws{PREFIX}"
print("base URL:", BASE_URL)
print("models  :", [VISION])

base URL: https://bedrock-mantle.us-east-1.api.aws/v1
models  : ['writer.palmyra-vision-7b']


## 1. First call

Auth is a short-term Bedrock API key minted from your ambient IAM credentials.
It expires within 12 hours and **cannot be refreshed** — mint a new one instead.
(`../00-foundations/01` shows the self-refreshing provider and the SigV4
alternative that needs no key at all.)

In [2]:
from aws_bedrock_token_generator import provide_token
from openai import OpenAI

# Build the client from a FRESH token — don't construct one at import time and
# reuse it for hours, because the baked-in key expires.
client = OpenAI(api_key=provide_token(region=REGION), base_url=BASE_URL)

completion = client.chat.completions.create(
    model=VISION,
    messages=[
        {
            "role": "user",
            "content": ("Explain what a vision-language model does, in two sentences."),
        }
    ],
    max_tokens=250,
)
# `content` can be None when the model spends the whole budget reasoning: the call
# succeeds with finish_reason="length" and no text. Check before printing -- this is
# the single most common surprise on this endpoint.
choice = completion.choices[0]
answer = choice.message.content or ""
if answer:
    print(answer)
else:
    print(f"(no text: finish_reason={choice.finish_reason!r} — raise max_tokens)")
print("\nusage:", completion.usage.model_dump_json())

 A vision-language model, such as clipGPT, uses deep learning to analyze both visual and textual information simultaneously, allowing it to understand and generate descriptions of images based on short text prompts. This model can bridge the gap between visual perception and language comprehension, enabling applications in tasks like image captioning, visual reasoning, and multimodal language processing.

usage: {"completion_tokens":70,"prompt_tokens":16,"total_tokens":86,"completion_tokens_details":null,"prompt_tokens_details":null}


## 2. Why Chat Completions and not Responses

AWS recommends the Responses API for new applications in general — but
availability is per-model. Probe both surfaces so the 400 is visible:

In [3]:
for api_name, path, body in [
    (
        "Chat Completions",
        f"{PREFIX}/chat/completions",
        {
            "model": VISION,
            "messages": [{"role": "user", "content": "Reply OK"}],
            "max_tokens": 16,
        },
    ),
    (
        "Responses (/v1)",
        f"{PREFIX}/responses",
        {"model": VISION, "input": "Reply OK", "max_output_tokens": 16},
    ),
    (
        "Responses (/openai/v1)",
        "/openai/v1/responses",
        {"model": VISION, "input": "Reply OK", "max_output_tokens": 16},
    ),
]:
    code, data = post(path, body, region=REGION)
    print(f"  {api_name:24} -> HTTP {code} {'' if code == 200 else err(data)[:64]}")

  Chat Completions         -> HTTP 200 


  Responses (/v1)          -> HTTP 400 The model 'writer.palmyra-vision-7b' does not support the '/v1/r


  Responses (/openai/v1)   -> HTTP 400 The model 'writer.palmyra-vision-7b' does not support the '/open


Concrete consequences of being Chat-Completions-only:

- **You own the conversation history.** There is no `previous_response_id`
  server-side state on this API — send the full `messages` array each turn.
- **Reasoning content is not returned.** `reasoning_effort` is accepted and the
  model does think, but the OpenAI Chat Completions schema has nowhere to put the
  trace, so you pay for those tokens without seeing them.
- Structured output uses `response_format`, not `text.format`.

## 3. Sampling parameters

This family accepts both `temperature` and `top_p`. That is *not* universal on
mantle — Gemma 4 rejects `top_p`, and Grok rejects `temperature` — so never share
one sampling config across families.

In [4]:
for label, extra in [
    ("temperature=0.7", {"temperature": 0.7}),
    ("temperature=0.0", {"temperature": 0.0}),
    ("top_p=0.95", {"top_p": 0.95}),
    ("both", {"temperature": 0.7, "top_p": 0.95}),
    ("max_tokens=1", {"max_tokens": 1}),
]:
    body = {
        "model": VISION,
        "messages": [{"role": "user", "content": "Reply OK"}],
        "max_tokens": 16,
    }
    body.update(extra)
    code, data = post(f"{PREFIX}/chat/completions", body, region=REGION)
    print(f"  {label:18} -> HTTP {code} {'' if code == 200 else err(data)[:60]}")

  temperature=0.7    -> HTTP 200 


  temperature=0.0    -> HTTP 200 


  top_p=0.95         -> HTTP 200 


  both               -> HTTP 200 


  max_tokens=1       -> HTTP 200 


Note `max_tokens=1` is accepted here. The Responses API enforces a minimum of
16 — another reason the two surfaces are not interchangeable.

## 4. Streaming

Chat Completions streams `data: {...}` SSE (server-sent events) frames carrying
`choices[0].delta.content`, terminated by `data: [DONE]`.

In [5]:
stream = client.chat.completions.create(
    model=VISION,
    messages=[
        {
            "role": "user",
            "content": ("List four business uses for document image understanding."),
        }
    ],
    max_tokens=300,
    stream=True,
)
chunks = 0
try:
    for chunk in stream:
        delta = chunk.choices[0].delta.content
        if delta:
            chunks += 1
            print(delta, end="", flush=True)
except Exception as exc:
    # A stream can fail AFTER delivering part of the answer: a mid-stream
    # 5xx is not rare, and it has happened while building these notebooks.
    # Report what arrived instead of losing it - production code has to
    # decide whether a partial answer is usable or the call must be retried.
    print(f"\n[stream interrupted after {chunks} deltas: {type(exc).__name__}]")
print(f"\n\n[{chunks} content deltas received]")

 Here are four business uses for document image understanding:

1. Improved efficiency in processing paper-based documents

2. Streamlined document management and retrieval

3. Enhanced compliance with Baghdadi RAND standards

4. Reduced time and costs associated with manual data entry

These applications demonstrate how document image understanding can bring significant benefits

 to businesses by automating document processing tasks. It allows for faster handling of large volumes of paperwork, ensures data integrity, and ultimately leads to a more efficient and organized operations.



[2 content deltas received]


## 5. Multi-turn — you manage the history

No server-side state on this API. Append each turn yourself.

In [6]:
# NOTE: this model requires strictly alternating user/assistant roles
# and rejects a leading `system` message with a 400. Put any instruction
# into the user turn instead.
messages = [
    {"role": "user", "content": "What is OCR?"},
]
first = client.chat.completions.create(model=VISION, messages=messages, max_tokens=200)
print("assistant:", first.choices[0].message.content)

messages.append({"role": "assistant", "content": first.choices[0].message.content})
messages.append(
    {
        "role": "user",
        "content": "How does a vision-language model differ from plain OCR?",
    }
)

second = client.chat.completions.create(model=VISION, messages=messages, max_tokens=200)
print("\nassistant:", second.choices[0].message.content)
print(
    f"\ninput tokens grew: {first.usage.prompt_tokens} -> {second.usage.prompt_tokens}"
)

assistant:  OCR stands for Optical Character Recognition. It's a technology that converts visual information, like text in images, into digital text that can be searched, edited, and used in various applications. OCR allows you to extract information from documents, signs, or other visual media containing text, making it easier to handle and analyzing the content digitally. This is particularly useful for digitizing old documents, transcribing images, or integrating text from various sources into digital systems.



assistant:  Vision-language models differ from plain OCR in several key ways:

1. Contextual understanding: Vision-language models have the ability to analyze both visual and textual information simultaneously, allowing them to understand the context and relationships between words and images.

2. semantics: While OCR simply extracts and digitizes text, vision-language models can infer meanings and concepts based on the visual context provided by the images and the text content.

3. multimodal learning: These models learn to integrate information from multiple sources, such as images and text, to produce more comprehensive and accurate results.

4. task-oriented: Vision-language models are designed to solve specific tasks that require both visual and textual processing, like question answering, image captioning, or commonsense reasoning.

5..* reduced roofs*, *屋*, *楼*, *面*, *积*, *描*, *燃*, *聚*, title: toilet harvests

This gives vision-language models a more sophisticated approach comp

That growth is the cost of client-side history. Families on the Responses API can
avoid it with `previous_response_id` (see `../03-google-gemma/`), at the price of
30-day server-side retention.

## 6. Reasoning effort

`reasoning_effort` is accepted. The trace is not returned — but the token count
moves, which is how you can tell the model really is thinking harder.

In [7]:
print(f"{'effort':10} {'status':>7} {'completion tokens':>18}")
print("-" * 38)
for effort in ("none", "low", "medium", "high"):
    code, data = post(
        f"{PREFIX}/chat/completions",
        {
            "model": VISION,
            "messages": [
                {
                    "role": "user",
                    "content": (
                        "If a chart shows sales of 10, 20 and 60 for three months, "
                        "what is "
                        "the growth rate month over month? Show your working."
                    ),
                }
            ],
            "max_tokens": 400,
            "reasoning_effort": effort,
        },
        region=REGION,
    )
    tokens = (data.get("usage") or {}).get("completion_tokens", "-")
    print(f"  {effort:8} {code:>7} {tokens!s:>18}")

effort      status  completion tokens
--------------------------------------


  none         200                154


  low          200                 94


  medium       200                214


  high         200                157


In [8]:
# THE LIMITATION, demonstrated rather than asserted: Palmyra Vision
# rejects tool definitions. Its model card lists "Client-side tool
# calling: Not Supported", and the API agrees.
probe_tool = [
    {
        "type": "function",
        "function": {
            "name": "noop",
            "description": "does nothing",
            "parameters": {
                "type": "object",
                "properties": {"x": {"type": "string"}},
                "required": ["x"],
            },
        },
    }
]

code, data = post(
    f"{PREFIX}/chat/completions",
    {
        "model": VISION,
        "messages": [{"role": "user", "content": "Call noop."}],
        "max_tokens": 64,
        "tools": probe_tool,
    },
    region=REGION,
)
print(f"tools on palmyra-vision -> HTTP {code}")
print("message:", err(data)[:150])
print("\n=> No forced-tool trick here. Use response_format instead (see below).")

tools on palmyra-vision -> HTTP 400
message: ErrorEvent { error: APIError { type: "BadRequestError", code: Some(400), message: "\"auto\" tool choice requires --enable-auto-tool-choice and --tool-

=> No forced-tool trick here. Use response_format instead (see below).


## 7. Does this model accept tool definitions?

Tool support is a per-model capability, so probe it rather than assuming either way.
At the time of writing the model card listed client-side tool calling as unsupported
and the API returned a 400, which the cell below shows.

If tools are rejected, the usual "force a tool call for strict JSON" trick is not
available, and `response_format` (next section) becomes the structured-output route.
Re-run this cell against your own model version before designing around either
answer.

## 8. Structured output with `response_format`

Two variants: loose `json_object`, and schema-enforced `json_schema`.

### Budget enough tokens, or you get nothing

A reasoning-capable model may spend most of its budget thinking before it emits
the opening brace. If `max_tokens` runs out first you get **HTTP 200 with empty
content** and `finish_reason="length"` - not an error, just nothing usable.
Always check `finish_reason` before parsing.

In [9]:
def json_object_call(prompt, max_tokens, model=VISION):
    code, data = post(
        f"{PREFIX}/chat/completions",
        {
            "model": model,
            "messages": [{"role": "user", "content": prompt}],
            "max_tokens": max_tokens,
            "response_format": {"type": "json_object"},
        },
        region=REGION,
    )
    choice = (data.get("choices") or [{}])[0]
    content = choice.get("message", {}).get("content") or ""
    return code, choice.get("finish_reason"), content


PROMPT = "Give the capital and population of France as JSON."
for budget in (64, 600):
    code, finish, content = json_object_call(PROMPT, budget)
    print(
        f"max_tokens={budget:4} HTTP {code} finish={finish!s:8} "
        f"content_len={len(content)}"
    )
    if finish == "length" and not content.strip():
        print("    -> truncated before any JSON was emitted; raise max_tokens")
    elif content.strip():
        # response_format json_object is a request, not a guarantee. This model
        # has been observed emitting a partial object and then STARTING AGAIN,
        # producing duplicated keys and no closing brace - which no parser can
        # rescue. Report that as the finding instead of letting it stop the
        # notebook, because an unreliable JSON mode is exactly what you need to
        # know before you depend on it.
        try:
            print("    ->", parse_json_lenient(content))
        except ValueError as exc:
            print(f"    -> UNPARSEABLE: {exc}")
            print(f"       raw ({len(content)} chars): {content[:150]!r}")
            print("       json_object mode did not produce valid JSON. Validate")
            print("       before use, and treat a parse failure as a normal path.")

print()
print("Valid JSON is not correct JSON. Compare the population across the two")
print("budgets above: this model has returned 67060681 and 67.4719458 for the")
print("same question. Both parse; one is not a population. Schema validation")
print("proves shape, never truth - range-check numeric fields yourself.")

max_tokens=  64 HTTP 200 finish=stop     content_len=50
    -> {'capital': 'Paris', 'population': 67.89252}


max_tokens= 600 HTTP 200 finish=stop     content_len=52
    -> {'capital': 'PARIS', 'population': '67000000'}

Valid JSON is not correct JSON. Compare the population across the two
budgets above: this model has returned 67060681 and 67.4719458 for the
same question. Both parse; one is not a population. Schema validation
proves shape, never truth - range-check numeric fields yourself.


In [10]:
schema = {
    "type": "object",
    "properties": {
        "country": {"type": "string"},
        "capital": {"type": "string"},
        "population_millions": {"type": "number"},
    },
    "required": ["country", "capital", "population_millions"],
    "additionalProperties": False,
}

code, data = post(
    f"{PREFIX}/chat/completions",
    {
        "model": VISION,
        "messages": [{"role": "user", "content": "Describe France."}],
        "max_tokens": 250,
        "response_format": {
            "type": "json_schema",
            "json_schema": {"name": "country", "strict": True, "schema": schema},
        },
    },
    region=REGION,
)
choice = (data.get("choices") or [{}])[0]
content = choice.get("message", {}).get("content")  # may be None!
print("json_schema ->", code, "| finish_reason:", choice.get("finish_reason"))
print("raw:", repr((content or "")[:160]))

if choice.get("finish_reason") == "length":
    # Reasoning consumed the budget before the object closed. Retry bigger.
    print("truncated - retrying with a larger budget")
    code, data = post(
        f"{PREFIX}/chat/completions",
        {
            "model": VISION,
            "messages": [{"role": "user", "content": "Describe France."}],
            "max_tokens": 2000,
            "response_format": {
                "type": "json_schema",
                "json_schema": {"name": "country", "strict": True, "schema": schema},
            },
        },
        region=REGION,
    )
    choice = (data.get("choices") or [{}])[0]
    content = choice.get("message", {}).get("content")
    print("retry finish_reason:", choice.get("finish_reason"))

parsed = parse_json_lenient(content or "")
print("parsed:", json.dumps(parsed, indent=2))
missing = {"country", "capital"} - set(parsed)
if missing:
    raise ValueError(f"model omitted required keys: {sorted(missing)} in {parsed}")
print("required keys present: country, capital")

json_schema -> 200 | finish_reason: length
raw: '{  \n"country": "France",  \n"capital": "Paris",  \n"population_millions": 67.8499999999999859291802117257861016471173796484485922376259649201813572121986737153559'
truncated - retrying with a larger budget


retry finish_reason: stop
parsed: {
  "country": "France",
  "capital": "Paris",
  "population_millions": 67.516
}
required keys present: country, capital


**Always parse leniently.** Even in strict mode, some mantle models append
characters after a valid object (Gemma 4 does this in ~half of runs), which makes
a bare `json.loads()` raise on output that is otherwise fine.

## 9. Compare the models in this family

Only one model in this family, so we compare prompting strategies instead of models.

In [11]:
task = (
    "In one sentence, when is a specialised vision model preferable to a general "
    "multimodal one?"
)

print(f"{'model':44} {'latency':>9} {'out tok':>8}  answer")
print("-" * 108)
for model in [VISION]:
    started = time.perf_counter()
    code, data = post(
        f"{PREFIX}/chat/completions",
        {
            "model": model,
            "messages": [{"role": "user", "content": task}],
            "max_tokens": 160,
        },
        region=REGION,
    )
    elapsed = time.perf_counter() - started
    if code != 200:
        print(f"{model:44} {'-':>9} {'-':>8}  HTTP {code}: {err(data)[:40]}")
        continue
    text = (data["choices"][0]["message"]["content"] or "").strip().replace("\n", " ")
    print(
        f"{model:44} {elapsed:>8.2f}s "
        f"{data['usage']['completion_tokens']:>8}  {text[:44]!r}"
    )

model                                          latency  out tok  answer
------------------------------------------------------------------------------------------------------------


writer.palmyra-vision-7b                         1.11s       58  'A specialised vision model is preferable to '


## 10. Latency: TTFT and throughput

TTFT is dominated by *prefill* (the model reading your prompt) plus queue time.
Service tiers trade cost against queue priority — they mostly separate under
contention, so single samples on an idle account look flat.
(`../00-foundations/03` has the full treatment.)

In [12]:
print(f"{'tier':10} {'TTFT (s)':>10} {'total (s)':>10} {'frames/s':>10}")
print("-" * 44)
for tier in ("default", "flex", "priority"):
    m = ttft(
        f"{PREFIX}/chat/completions",
        {
            "model": VISION,
            "messages": [
                {
                    "role": "user",
                    "content": (
                        "List four business uses for document image understanding."
                    ),
                }
            ],
            "max_tokens": 200,
            "service_tier": tier,
        },
        region=REGION,
    )
    if m.get("error"):
        print(f"{tier:10} {m['error']:>32}  (tier not supported by this model)")
    else:
        print(
            f"{tier:10} {m['ttft_s']:>10.3f} {m['total_s']:>10.3f} "
            f"{m['frames_per_s']:>10.1f}"
        )

tier         TTFT (s)  total (s)   frames/s
--------------------------------------------


default         0.966      1.485        9.6


flex            1.292      1.306      139.4


priority        0.939      1.161       18.0


## 11. Production hardening

Retries, cost attribution, and privacy. Mantle has **no RPM quota** — throttling
is token-based, and most models here have no published TPM (tokens per minute) quota
at all, so
capacity is fair-share. That makes retry-with-backoff mandatory, not optional.

In [13]:
code, project = post(
    "/v1/organization/projects",
    {
        "name": "palmyra-samples",
        "tags": {"Application": "PalmyraDemo", "Environment": "Demo"},
    },
    region=REGION,
)
project_id = project.get("id")
safe_print("project:", code, project_id)

code, data = post(
    f"{PREFIX}/chat/completions",
    {
        "model": VISION,
        "messages": [{"role": "user", "content": "Reply OK"}],
        "max_tokens": 16,
    },
    region=REGION,
    headers={"OpenAI-Project": project_id},  # cost attribution
)
print("attributed call ->", code)

project: 200 proj_qahopa3t...


attributed call -> 200


In [14]:
class PalmyraClient:
    """Demonstrates retry, attribution and structured-output patterns for this
    family on bedrock-mantle. A teaching pattern, not a production component:
    review and adapt it, and have it security-reviewed, before deployment."""

    def __init__(self, model=VISION, region=REGION, tier="default", project=None):
        self.model, self.region, self.tier, self.project = model, region, tier, project

    def chat(self, messages, *, max_tokens=512, schema=None, effort=None):
        # NOTE: no `tools` param (rejected) and no `system` role (rejected).
        body = {
            "model": self.model,
            "messages": messages,
            "max_tokens": max_tokens,
            "temperature": 0.7,
            "service_tier": self.tier,
        }
        if effort:
            body["reasoning_effort"] = effort
        if schema:
            body["response_format"] = {
                "type": "json_schema",
                "json_schema": {"name": "out", "strict": True, "schema": schema},
            }
        headers = {"OpenAI-Project": self.project} if self.project else None
        # post() retries 429 + 5xx with exponential backoff and jitter.
        code, data = post(
            f"{PREFIX}/chat/completions", body, region=self.region, headers=headers
        )
        if code != 200:
            raise RuntimeError(f"HTTP {code}: {err(data)}")
        return data

    @staticmethod
    def _choice(data: dict) -> dict:
        """First choice, without assuming the list is non-empty."""
        return (data.get("choices") or [{}])[0]

    def json(self, prompt, schema, *, max_tokens=512, **kw):
        """Structured call that survives both failure modes seen on this model.

        1. EMPTY content with HTTP 200 and finish_reason="length" - a
           reasoning-capable model can spend the whole budget thinking. Parsing
           that raises, so escalate the budget and retry.
        2. NON-EMPTY content that is not valid JSON. `response_format` is a
           request, not a guarantee: this model has been observed emitting a
           partial object and then starting again, giving duplicated keys and no
           closing brace. parse_json_lenient() repairs a merely truncated object
           but cannot rescue that, so treat a parse failure as a retryable
           outcome rather than letting it escape to the caller.

        Raises RuntimeError only after every attempt is exhausted, so a caller
        never has to distinguish "empty" from "malformed".
        """
        messages = [{"role": "user", "content": prompt}]
        last = "no attempt made"
        for budget in (max_tokens, max_tokens * 4):
            data = self.chat(messages, schema=schema, max_tokens=budget, **kw)
            choice = self._choice(data)
            content = choice.get("message", {}).get("content") or ""
            if content.strip():
                try:
                    return parse_json_lenient(content)
                except ValueError as exc:
                    last = f"unparseable at {budget} tokens: {exc}"
                    continue  # a bigger budget sometimes yields a clean object
            if choice.get("finish_reason") != "length":
                last = f"empty, finish_reason={choice.get('finish_reason')!r}"
                break  # empty for some other reason - escalating will not help
            last = f"empty and truncated at {budget} tokens"
        raise RuntimeError(f"no usable JSON after escalation: {last}")


bot = PalmyraClient(tier="flex", project=project_id)
out = bot.json(
    "Name the largest ocean and its average depth in metres.",
    {
        "type": "object",
        "properties": {"ocean": {"type": "string"}, "avg_depth_m": {"type": "number"}},
        "required": ["ocean", "avg_depth_m"],
        "additionalProperties": False,
    },
)
print("structured result:", out)

structured result: {'ocean': ' Pacific Ocean', 'avg_depth_m': 4000}


In [15]:
code, archived = post(
    f"/v1/organization/projects/{project_id}/archive", {}, region=REGION
)
print("archived demo project:", code, archived.get("status"))

archived demo project: 200 archived


## Gotchas — Writer Palmyra Vision on bedrock-mantle

| Gotcha | Detail |
|---|---|
| Path prefix | Bare `/v1`, **not** `/openai/v1` (that's gemma-4 / gpt-5.x / grok) |
| Responses API | Returns **400** for this family — Chat Completions only |
| History | No `previous_response_id`; you send `messages` every turn |
| Reasoning trace | `reasoning_effort` works but the trace is never returned |
| Strict JSON | Parse leniently — models can append text after a valid object |
| Sampling | `temperature` **and** `top_p` both fine here; not true family-wide |
| `max_tokens` | 1 is valid here; Responses API demands ≥16 |
| `content` can be `None` | Check `finish_reason` before slicing/parsing content |
| Quotas | No RPM quota; most models have no published TPM — retry with backoff |
| `reserved` tier | Rejected as a parameter; arranged via your account team |
| CloudWatch | Metrics land in `AWS/BedrockMantle`, not `AWS/Bedrock` |
| **Tool calling** | **Rejected with 400** — matches the model card |
| Structured output | `response_format` works; use it instead of forced tools |
| **`system` role** | **Rejected** — roles must strictly alternate user/assistant |

## Where next
- Same API shape: `../04-qwen/` (qwen3-vl), `../10-nvidia-nemotron/` (nano-12b-v2)
- Different API shape: `../03-google-gemma/` (Responses),
  `../02-anthropic-claude/` (Messages), `../01-openai-gpt/` (web search, caching)
- Shared mechanics: `../00-foundations/`

## Also on `bedrock-runtime`? Palmyra

`palmyra-vision-7b` is on both endpoints. Writer's text models `palmyra-x4` and `palmyra-x5` are `bedrock-runtime` only, so Converse is the *only* way to reach them.

`endpoints_for()` asks both catalogues rather than trusting a table, so the cell
below tells you today's answer. Converse is worth reaching for when you want one
request shape across providers, or a feature that only `bedrock-runtime` carries.


In [16]:
from bedrock import (
    converse,
    converse_reasoning,
    endpoints_for,
    resolve_runtime_id,
)

MANTLE_ID = "writer.palmyra-vision-7b"
RUNTIME_ID = "writer.palmyra-vision-7b"

print("endpoint availability:", endpoints_for(MANTLE_ID))
print("mantle model id :", MANTLE_ID)
print("runtime model id:", RUNTIME_ID)
resolved = resolve_runtime_id(RUNTIME_ID)
print("converse sends  :", resolved)
if resolved != RUNTIME_ID:
    print("                  ^ resolved for you; the form above would be rejected")

# The same question, through Converse. Note the shape: content is a LIST of
# blocks rather than a string, and the token budget lives in inferenceConfig.
# The budget is generous on purpose - a reasoning model spends it on the trace
# first and returns no text block at all if it runs out.
text, response = converse(
    RUNTIME_ID,
    [
        {
            "role": "user",
            "content": [
                {"text": "Describe, in one sentence, what a bar chart is for."}
            ],
        }
    ],
    max_tokens=400,
    # NOTE: writer.palmyra-vision-7b REJECTS a system prompt on Converse -
    # passing `system=` returns ValidationException. Verified in us-east-1.
    # Fold your instructions into the user turn for this model.
)

error = (response.get("error") or {}).get("message")
if error:
    print("\ncall failed:", error[:200])
else:
    reasoning = converse_reasoning(response)
    print("\nstop reason:", response.get("stopReason"))
    print("tokens     :", response.get("usage", {}).get("totalTokens"))
    if reasoning:
        print(f"reasoning  : {len(reasoning)} chars (returned in a "
              "reasoningContent block, before the text)")
    if text.strip():
        print("answer     :", text.strip()[:200])
        print()
        print("Palmyra Vision is specialised for image understanding. Asked a")
        print("text-only question well outside that remit it stays fluent but")
        print("drifts off-topic - ask it about an abstract systems concept and it")
        print("will happily answer about skincare. Judge it on the vision tasks")
        print("earlier in this notebook, not on text-only reasoning.")
    else:
        # Empty text is NOT the same as a failed call. Say which it is.
        print("answer     : (none - the budget went to reasoning; raise max_tokens)")


endpoint availability: {'mantle': True, 'runtime': True}
mantle model id : writer.palmyra-vision-7b
runtime model id: writer.palmyra-vision-7b


converse sends  : writer.palmyra-vision-7b



stop reason: end_turn
tokens     : 41
answer     : A bar chart displays data using vertical bars of varying heights, providing a clear visual comparison of different categories or values.

Palmyra Vision is specialised for image understanding. Asked a
text-only question well outside that remit it stays fluent but
drifts off-topic - ask it about an abstract systems concept and it
will happily answer about skincare. Judge it on the vision tasks
earlier in this notebook, not on text-only reasoning.


## Converse in earnest — the tool loop, provider parameters, and caching

The earlier endpoint section proved this model answers through Converse. That is
the easy part. This section does the three things you actually need on
`bedrock-runtime`, because each differs from the `bedrock-mantle` equivalent:

1. **A complete tool round trip** — `toolUse` out, `toolResult` back in. Getting a
   tool *call* is half the job; feeding the result back is where the shapes bite.
2. **`additionalModelRequestFields`** — Converse normalises the common fields, so
   anything provider-specific goes through this escape hatch.
3. **`cachePoint`** — prompt caching is a first-class Converse block, and support
   for it is per model rather than universal.


In [17]:
from bedrock import converse_text, resolve_runtime_id, runtime_client

RUNTIME_ID = "writer.palmyra-vision-7b"
runtime = runtime_client(REGION)
resolved = resolve_runtime_id(RUNTIME_ID, REGION)

WEATHER_TOOL = {
    "toolSpec": {
        "name": "get_weather",
        "description": "Current weather for a city",
        "inputSchema": {
            "json": {
                "type": "object",
                "properties": {"city": {"type": "string"}},
                "required": ["city"],
            }
        },
    }
}

# This model rejected tool definitions on bedrock-mantle earlier in the notebook.
# Converse is a different API, so the question is worth asking again rather than
# assumed - the answer is a property of the model, not of the endpoint.
try:
    runtime.converse(
        modelId=resolved,
        messages=[{"role": "user", "content": [{"text": "Weather in Singapore? Use the tool."}]}],
        toolConfig={"tools": [WEATHER_TOOL]},
        inferenceConfig={"maxTokens": 300},
    )
    print("toolConfig accepted on Converse")
except Exception as exc:
    print(f"toolConfig -> {type(exc).__name__}")
    print(f"    {str(exc)[-150:]}")
    print()
    print("Same answer on both endpoints, so this is the model's capability rather")
    print("than an endpoint limitation. Plan for prompt-and-parse instead of tools.")

# The plain call still works, so the model is usable - just not with tools.
text, response = converse(
    RUNTIME_ID,
    [{"role": "user", "content": [{"text": "Describe a bar chart in one sentence."}]}],
    max_tokens=120,
    region=REGION,
)
print()
print("plain Converse call:", (text or "").strip()[:120])


toolConfig -> ValidationException
    \\\" tool choice requires --enable-auto-tool-choice and --tool-call-parser to be set\", param: None } }","param":null,"type":"invalid_request_error"}}

Same answer on both endpoints, so this is the model's capability rather
than an endpoint limitation. Plan for prompt-and-parse instead of tools.



plain Converse call: A bar chart displays data vertically using rectangular bars of varying heights, where the length of each bar represents 


In [18]:
# cachePoint is a Converse block, but support for it is per model rather than
# universal. Ask before designing around it.
HANDBOOK = "You are a support handbook. " + (
    "Retries: use exponential backoff with full jitter, cap at 16 seconds. " * 160
)

try:
    usage = runtime.converse(
        modelId=resolved,
        system=[{"text": HANDBOOK}, {"cachePoint": {"type": "default"}}],
        messages=[{"role": "user", "content": [{"text": "One line: the retry policy?"}]}],
        inferenceConfig={"maxTokens": 60},
    )["usage"]
    print("cachePoint accepted:", {k: v for k, v in usage.items() if "cache" in k.lower()})
except Exception as exc:
    print(f"cachePoint -> {type(exc).__name__}")
    print(f"    {str(exc)[-140:]}")
    print()
    print("Not every model supports prompt caching on Converse. Note the exception")
    print("type: this surfaces as an access or validation error rather than a clear")
    print("'unsupported feature' message, which is easy to misread as a permissions")
    print("problem. Probe it once per model instead of assuming it is available.")

# The same call without the cachePoint block works, so caching is the only part
# that is unavailable.
#
# Note there is NO system= here. This model rejects a system prompt on Converse
# (proved in the endpoint section above), so the instruction goes in the user turn.
# Two separate per-model limitations stack on the same call, which is exactly why
# each one has to be probed rather than inferred from the family.
usage = runtime.converse(
    modelId=resolved,
    messages=[
        {
            "role": "user",
            "content": [{"text": "Be terse. One line: why use backoff?"}],
        }
    ],
    inferenceConfig={"maxTokens": 60},
)["usage"]
print()
print("same call without cachePoint or system -> OK, tokens:", usage["totalTokens"])


cachePoint -> AccessDeniedException
    nverse operation: You invoked an unsupported model or your request did not allow prompt caching. See the documentation for more information.

Not every model supports prompt caching on Converse. Note the exception
type: this surfaces as an access or validation error rather than a clear
'unsupported feature' message, which is easy to misread as a permissions
problem. Probe it once per model instead of assuming it is available.



same call without cachePoint or system -> OK, tokens: 25


### What this section adds over the endpoint check above

- **The tool loop is the part that bites.** `toolSpec` is not the OpenAI shape,
  the JSON Schema nests under `inputSchema.json`, and the second turn must echo the
  assistant message back verbatim alongside a `toolResult` whose `toolUseId`
  matches. Miss any of that and you get a 400.
- **`additionalModelRequestFields` is unvalidated by Converse.** It is the only way
  to reach provider-specific behaviour, and a key the model does not recognise
  fails the call rather than being ignored.
- **Feature support is per model, not per endpoint.** Read the output above rather
  than carrying an assumption over from another family.
